# B3 verified-patch v1 — test-only Devign + REVEAL

Prepared for Kaggle account `duchuynguyen1802`. Source: canonical B3 notebook from `lznt123`.
Test labels stay local. This notebook runs inference only; it does not compute test metrics. Runtime cap: 10 hours.


In [1]:
from pathlib import Path
import base64, hashlib, json, os, sys, zipfile, shutil, subprocess, time, platform
from datetime import datetime, timezone
WORK=Path('/kaggle/working'); WORK.mkdir(exist_ok=True)
SESSION_STARTED=time.time()
for name, encoded in {'grace_runtime_contract.py': 'aW1wb3J0IGNvbnRleHRsaWIKaW1wb3J0IGhhc2hsaWIKaW1wb3J0IGltcG9ydGxpYi5tZXRhZGF0YQppbXBvcnQganNvbgppbXBvcnQgbXVsdGlwcm9jZXNzaW5nCmltcG9ydCBvcwppbXBvcnQgcGxhdGZvcm0KaW1wb3J0IHJlCmltcG9ydCBzaHV0aWwKaW1wb3J0IHN1YnByb2Nlc3MKaW1wb3J0IHN5cwpmcm9tIHBhdGhsaWIgaW1wb3J0IFBhdGgKCkVYUEVDVEVEX0NPTlRSQUNUX1NIQTI1NiA9ICdiZjFmODU5ZGM3ODZlMjA2YTNjMTFkNjU1YjA0YjUyOGRjNjhmNTlhMTVhODJkNGRiODk0YWM1OTQ0YjBhYzJlJwpDT05UUkFDVF9QQVRIID0gUGF0aCgnL2thZ2dsZS93b3JraW5nL2luZmVyZW5jZV9ydW50aW1lX2xvY2suanNvbicpCgpkZWYgZGlnZXN0KHBhdGgpOgogICAgaCA9IGhhc2hsaWIuc2hhMjU2KCkKICAgIHdpdGggUGF0aChwYXRoKS5vcGVuKCdyYicpIGFzIGY6CiAgICAgICAgZm9yIGJsb2NrIGluIGl0ZXIobGFtYmRhOiBmLnJlYWQoMTAyNCAqIDEwMjQpLCBiJycpOiBoLnVwZGF0ZShibG9jaykKICAgIHJldHVybiBoLmhleGRpZ2VzdCgpCgpkZWYgbG9hZF9jb250cmFjdChwYXRoPUNPTlRSQUNUX1BBVEgpOgogICAgcmF3ID0gUGF0aChwYXRoKS5yZWFkX2J5dGVzKCkKICAgIGlmIGhhc2hsaWIuc2hhMjU2KHJhdykuaGV4ZGlnZXN0KCkgIT0gRVhQRUNURURfQ09OVFJBQ1RfU0hBMjU2OgogICAgICAgIHJhaXNlIFZhbHVlRXJyb3IoJ1J1bnRpbWUgY29udHJhY3QgZmlsZSBTSEEtMjU2IG1pc21hdGNoJykKICAgIGMgPSBqc29uLmxvYWRzKHJhdy5kZWNvZGUoJ3V0Zi04JykpCiAgICB1bnNpZ25lZCA9IHtrOiB2IGZvciBrLCB2IGluIGMuaXRlbXMoKSBpZiBrICE9ICdpbnRlZ3JpdHknfQogICAgc2VhbCA9IGhhc2hsaWIuc2hhMjU2KGpzb24uZHVtcHModW5zaWduZWQsIHNvcnRfa2V5cz1UcnVlLCBzZXBhcmF0b3JzPSgnLCcsICc6JyksIGVuc3VyZV9hc2NpaT1GYWxzZSkuZW5jb2RlKCd1dGYtOCcpKS5oZXhkaWdlc3QoKQogICAgaWYgc2VhbCAhPSBjWydpbnRlZ3JpdHknXVsnc2hhMjU2J106CiAgICAgICAgcmFpc2UgVmFsdWVFcnJvcignUnVudGltZSBjb250cmFjdCBjYW5vbmljYWwgaW50ZWdyaXR5IG1pc21hdGNoJykKICAgIHJldHVybiBjCgpkZWYgY29tbWFuZChhcmdzKToKICAgIHAgPSBzdWJwcm9jZXNzLnJ1bihhcmdzLCBjaGVjaz1UcnVlLCB0ZXh0PVRydWUsIGNhcHR1cmVfb3V0cHV0PVRydWUpCiAgICByZXR1cm4gcC5zdGRvdXQgKyBwLnN0ZGVycgoKZGVmIGFzc2VydF9ydW50aW1lKGMpOgogICAgYWN0dWFsID0ge25hbWU6IGltcG9ydGxpYi5tZXRhZGF0YS52ZXJzaW9uKG5hbWUpIGZvciBuYW1lIGluIGNbJ3BhY2thZ2VzJ119CiAgICBpZiBhY3R1YWwgIT0gY1sncGFja2FnZXMnXTogcmFpc2UgUnVudGltZUVycm9yKGYnUGFja2FnZSBtaXNtYXRjaDoge2FjdHVhbH0nKQogICAgaWYgcGxhdGZvcm0uc3lzdGVtKCkgIT0gJ0xpbnV4JyBvciAnLicuam9pbihtYXAoc3RyLCBzeXMudmVyc2lvbl9pbmZvWzoyXSkpICE9IGNbJ2J1aWxkJ11bJ3B5dGhvbl9mYW1pbHknXToKICAgICAgICByYWlzZSBSdW50aW1lRXJyb3IoJ0NvbnRyYWN0IHJlcXVpcmVzIEthZ2dsZSBMaW51eC9QeXRob24gMy4xMycpCiAgICBncHUgPSBjb21tYW5kKFsnbnZpZGlhLXNtaScsICctLXF1ZXJ5LWdwdT1pbmRleCx1dWlkLG5hbWUsbWVtb3J5LmZyZWUsbWVtb3J5LnRvdGFsLGRyaXZlcl92ZXJzaW9uJywgJy0tZm9ybWF0PWNzdixub2hlYWRlciddKQogICAgbGluZXMgPSBbcyBmb3IgcyBpbiBncHUuc3BsaXRsaW5lcygpIGlmIHMuc3RyaXAoKV0KICAgIGlmIGxlbihsaW5lcykgIT0gY1snZ3B1J11bJ2RldmljZV9jb3VudCddIG9yIGFueSgnVDQnIG5vdCBpbiBzIGZvciBzIGluIGxpbmVzKToKICAgICAgICByYWlzZSBSdW50aW1lRXJyb3IoZidDb250cmFjdCByZXF1aXJlcyAyIFQ0IEdQVXM6IHtncHV9JykKICAgIGRpc3QgPSBpbXBvcnRsaWIubWV0YWRhdGEuZGlzdHJpYnV0aW9uKCdsbGFtYS1jcHAtcHl0aG9uJykKICAgIG9yaWdpbiA9IGpzb24ubG9hZHMoZGlzdC5yZWFkX3RleHQoJ2RpcmVjdF91cmwuanNvbicpIG9yICd7fScpCiAgICBpZiBvcmlnaW4uZ2V0KCdhcmNoaXZlX2luZm8nLCB7fSkuZ2V0KCdoYXNoZXMnLCB7fSkuZ2V0KCdzaGEyNTYnKSAhPSBjWydiYWNrZW5kX3NvdXJjZSddWydzZGlzdF9zaGEyNTYnXToKICAgICAgICByYWlzZSBSdW50aW1lRXJyb3IoJ0JhY2tlbmQgaW5zdGFsbGF0aW9uIGlzIG5vdCBmcm9tIHRoZSB2ZXJpZmllZCBzb3VyY2UgYXJjaGl2ZScpCiAgICByZXR1cm4gYWN0dWFsLCBncHUKCmRlZiBwcm92ZW5hbmNlKGMsIHJ1bm5lciwgdGFyZ2V0KToKICAgIHBhY2thZ2VzLCBncHUgPSBhc3NlcnRfcnVudGltZShjKQogICAgZGlzdCA9IGltcG9ydGxpYi5tZXRhZGF0YS5kaXN0cmlidXRpb24oJ2xsYW1hLWNwcC1weXRob24nKQogICAgYmluYXJpZXMgPSB7fQogICAgZm9yIGVudHJ5IGluIGRpc3QuZmlsZXMgb3IgW106CiAgICAgICAgaWYgc3RyKGVudHJ5KS5lbmRzd2l0aCgoJy5zbycsICcuZGxsJywgJy5keWxpYicpKToKICAgICAgICAgICAgcCA9IFBhdGgoZGlzdC5sb2NhdGVfZmlsZShlbnRyeSkpOyBiaW5hcmllc1tzdHIoZW50cnkpXSA9IGRpZ2VzdChwKQogICAgaWYgbm90IGJpbmFyaWVzOiByYWlzZSBSdW50aW1lRXJyb3IoJ05vIGNvbXBpbGVkIGJhY2tlbmQgbGlicmFyeSBmb3VuZCBmb3IgcHJvdmVuYW5jZScpCiAgICByZXN1bHQgPSBkaWN0KHJ1bnRpbWVfY29udHJhY3Rfc2hhMjU2PUVYUEVDVEVEX0NPTlRSQUNUX1NIQTI1NiwKICAgICAgICBydW50aW1lX2NvbnRyYWN0X2ludGVncml0eT1jWydpbnRlZ3JpdHknXVsnc2hhMjU2J10sIHBhY2thZ2VzPXBhY2thZ2VzLAogICAgICAgIHB5dGhvbj1zeXMudmVyc2lvbiwgcGxhdGZvcm09cGxhdGZvcm0ucGxhdGZvcm0oKSwgbGliYz1wbGF0Zm9ybS5saWJjX3ZlcigpLAogICAgICAgIGdwdV9iZWZvcmVfbG9hZD1ncHUsIGNwdV9jb3VudD1tdWx0aXByb2Nlc3NpbmcuY3B1X2NvdW50KCksCiAgICAgICAgbl90aHJlYWRzPW1heChtdWx0aXByb2Nlc3NpbmcuY3B1X2NvdW50KCkgLy8gMiwgMSksIG5fdGhyZWFkc19iYXRjaD1tdWx0aXByb2Nlc3NpbmcuY3B1X2NvdW50KCksCiAgICAgICAgY29uc3RydWN0b3I9Y1snY29uc3RydWN0b3InXSwgcnVubmVyX3NoYTI1Nj1kaWdlc3QocnVubmVyKSwKICAgICAgICBjdWRhX2RldmljZV9vcmRlcj1vcy5lbnZpcm9uLmdldCgnQ1VEQV9ERVZJQ0VfT1JERVInKSwKICAgICAgICBjdWRhX3Zpc2libGVfZGV2aWNlcz1vcy5lbnZpcm9uLmdldCgnQ1VEQV9WSVNJQkxFX0RFVklDRVMnKSwKICAgICAgICBiYWNrZW5kX3NkaXN0X3NoYTI1Nj1jWydiYWNrZW5kX3NvdXJjZSddWydzZGlzdF9zaGEyNTYnXSwgY29tcGlsZWRfYmFja2VuZF9zaGEyNTY9YmluYXJpZXMsCiAgICAgICAgYnVpbGRfZW52aXJvbm1lbnQ9e2s6IG9zLmVudmlyb24uZ2V0KGspIGZvciBrIGluICgnQ01BS0VfQVJHUycsICdDTUFLRV9CVUlMRF9QQVJBTExFTF9MRVZFTCcsICdGT1JDRV9DTUFLRScpfSkKICAgIGZvciBuYW1lLCBhcmdzIGluIFsoJ252Y2MnLCBbJ252Y2MnLCAnLS12ZXJzaW9uJ10pLCAoJ2NvbXBpbGVyJywgWydjYycsICctLXZlcnNpb24nXSksCiAgICAgICAgICAgICAgICAgICAgICAoJ2N4eF9jb21waWxlcicsIFsnYysrJywgJy0tdmVyc2lvbiddKSwgKCdjbWFrZScsIFsnY21ha2UnLCAnLS12ZXJzaW9uJ10pLAogICAgICAgICAgICAgICAgICAgICAgKCdwaXBfZnJlZXplJywgW3N5cy5leGVjdXRhYmxlLCAnLW0nLCAncGlwJywgJ2ZyZWV6ZScsICctLWFsbCddKV06CiAgICAgICAgcmVzdWx0W25hbWVdID0gY29tbWFuZChhcmdzKQogICAgdGFyZ2V0ID0gUGF0aCh0YXJnZXQpOyB0YXJnZXQucGFyZW50Lm1rZGlyKHBhcmVudHM9VHJ1ZSwgZXhpc3Rfb2s9VHJ1ZSkKICAgIHRhcmdldC53cml0ZV90ZXh0KGpzb24uZHVtcHMocmVzdWx0LCBpbmRlbnQ9MiksIGVuY29kaW5nPSd1dGYtOCcpCiAgICByZXR1cm4gcmVzdWx0CgpkZWYgbG9hZF9sb2NrZWRfbW9kZWwoYywgbW9kZWxfcGF0aCwgdG9rZW5pemVyX21hbmlmZXN0LCBncHVfbGF5ZXJzLCBsb2dfcGF0aCwgbW9kZWxfZGlnZXN0KToKICAgIGltcG9ydCBsbGFtYV9jcHAKICAgIGlmIG1vZGVsX2RpZ2VzdCAhPSBjWydtb2RlbCddWydzaGEyNTYnXTogcmFpc2UgVmFsdWVFcnJvcignRnJvemVuIG1vZGVsIFNIQS0yNTYgbWlzbWF0Y2gnKQogICAgaWYgZ3B1X2xheWVycyAhPSBjWydjb25zdHJ1Y3RvciddWyduX2dwdV9sYXllcnMnXTogcmFpc2UgVmFsdWVFcnJvcignR1BVIGxheWVycyBkaWZmZXIgZnJvbSBsb2NrJykKICAgIGlmICh0b2tlbml6ZXJfbWFuaWZlc3RbJ3Rva2VuaXplciddLCB0b2tlbml6ZXJfbWFuaWZlc3RbJ3Rva2VuaXplcl9yZXZpc2lvbiddLCB0b2tlbml6ZXJfbWFuaWZlc3RbJ3Byb21wdF9idWRnZXRfdG9rZW5zJ10pICE9ICgKICAgICAgICBjWyd0b2tlbml6ZXInXVsncmVwbyddLCBjWyd0b2tlbml6ZXInXVsncmV2aXNpb24nXSwgY1snY29uc3RydWN0b3InXVsnbl9jdHgnXSk6CiAgICAgICAgcmFpc2UgVmFsdWVFcnJvcignVG9rZW5pemVyL2NvbnRleHQgZGlmZmVyIGZyb20gcnVudGltZSBsb2NrJykKICAgIGlmIG5vdCBsbGFtYV9jcHAubGxhbWFfc3VwcG9ydHNfZ3B1X29mZmxvYWQoKTogcmFpc2UgUnVudGltZUVycm9yKCdDVURBIGJhY2tlbmQgdW5hdmFpbGFibGUnKQogICAga3dhcmdzID0gZGljdChjWydjb25zdHJ1Y3RvciddKQogICAga3dhcmdzWydzcGxpdF9tb2RlJ10gPSBnZXRhdHRyKGxsYW1hX2NwcCwga3dhcmdzWydzcGxpdF9tb2RlJ10pCiAgICBrd2FyZ3MudXBkYXRlKG5fdGhyZWFkcz1tYXgobXVsdGlwcm9jZXNzaW5nLmNwdV9jb3VudCgpIC8vIDIsIDEpLCBuX3RocmVhZHNfYmF0Y2g9bXVsdGlwcm9jZXNzaW5nLmNwdV9jb3VudCgpKQogICAgcGF0aCA9IFBhdGgobG9nX3BhdGgpCiAgICB3aXRoIHBhdGgub3BlbigndycsIGVuY29kaW5nPSd1dGYtOCcpIGFzIGxvZywgY29udGV4dGxpYi5yZWRpcmVjdF9zdGRlcnIobG9nKToKICAgICAgICBtb2RlbCA9IGxsYW1hX2NwcC5MbGFtYShtb2RlbF9wYXRoPXN0cihtb2RlbF9wYXRoKSwgdmVyYm9zZT1UcnVlLCAqKmt3YXJncykKICAgIHRleHQgPSBwYXRoLnJlYWRfdGV4dChlbmNvZGluZz0ndXRmLTgnLCBlcnJvcnM9J3JlcGxhY2UnKQogICAgcHJpbnQodGV4dCwgZmx1c2g9VHJ1ZSkKICAgIGZyb20gYjNfbmF0aXZlX3J1bnRpbWUgaW1wb3J0IG5hdGl2ZV9sb2dfY2hlY2sKICAgIHRyeToKICAgICAgICBuYXRpdmVfbG9nX2NoZWNrKHRleHQsIGMpCiAgICBleGNlcHQgRXhjZXB0aW9uOgogICAgICAgIG1vZGVsLmNsb3NlKCkKICAgICAgICByYWlzZQogICAgcmV0dXJuIG1vZGVsCg==', 'b3_native_runtime.py': 'aW1wb3J0IHJlCgpkZWYgbmF0aXZlX2xvZ19jaGVjayh0ZXh0LCBjb250cmFjdCk6CiAgICByZXF1aXJlZCA9IGNvbnRyYWN0WydncHUnXVsncmVxdWlyZWRfb2ZmbG9hZGVkX2xheWVycyddCiAgICBwYWlycyA9IHJlLmZpbmRhbGwocidvZmZsb2FkZWRccysoXGQrKVxzKi9ccyooXGQrKVxzK2xheWVycz9ccyt0b1xzK0dQVScsIHRleHQsIHJlLkkpCiAgICBpZiBub3QgYW55KGludChhKSA9PSBpbnQoYikgPT0gcmVxdWlyZWQgZm9yIGEsYiBpbiBwYWlycyk6CiAgICAgICAgcmFpc2UgVmFsdWVFcnJvcignRnVsbCBHUFUgb2ZmbG9hZCBub3QgdmVyaWZpZWQnKQogICAgZm9yIGZpZWxkIGluICgnbl9jdHgnLCduX2JhdGNoJywnbl91YmF0Y2gnKToKICAgICAgICB2YWx1ZXMgPSByZS5maW5kYWxsKHInXGInK2ZpZWxkK3InXHMqPVxzKihcZCspXGInLCB0ZXh0KQogICAgICAgIGlmIG5vdCB2YWx1ZXMgb3IgaW50KHZhbHVlc1stMV0pICE9IGNvbnRyYWN0Wydjb25zdHJ1Y3RvciddW2ZpZWxkXToKICAgICAgICAgICAgcmFpc2UgVmFsdWVFcnJvcignRWZmZWN0aXZlICcrZmllbGQrJyBkaWZmZXJzIGZyb20gY29udHJhY3QnKQogICAgZGV2aWNlcyA9IHNldChyZS5maW5kYWxsKHIndXNpbmcgZGV2aWNlIENVREEoXGQrKVxzKlwoW15cbildKlxiVDRcYlteXG4pXSpcKScsIHRleHQsIHJlLkkpKQogICAgZGV2aWNlcy51cGRhdGUocmUuZmluZGFsbChyJ0RldmljZVxzKyhcZCspOlxzK1teXG5dKlxiVDRcYicsIHRleHQpKQogICAgaWYgZGV2aWNlcyAhPSBzZXQobWFwKHN0ciwgcmFuZ2UoY29udHJhY3RbJ2dwdSddWydkZXZpY2VfY291bnQnXSkpKToKICAgICAgICByYWlzZSBWYWx1ZUVycm9yKCdUd28gZGlzdGluY3QgVDQgZGV2aWNlcyBub3QgdmVyaWZpZWQnKQo=', 'inference_runtime_lock.json': '...', 'b3_test_runtime.py': 'IiIiRmFpbC1jbG9zZWQgQjMgdGVzdCBwYWNrYWdlIGxvYWRlci4gVGFyZ2V0IGxhYmVscyBuZXZlciBlbnRlciB0aGlzIG1vZHVsZS4iIiIKaW1wb3J0IGhhc2hsaWIKaW1wb3J0IGpzb24KaW1wb3J0IG1hdGgKaW1wb3J0IHJlCmltcG9ydCB6aXBmaWxlCmZyb20gcGF0aGxpYiBpbXBvcnQgUGF0aCwgUHVyZVBvc2l4UGF0aAoKCmRlZiBzaGEocmF3KToKICAgIGlmIGlzaW5zdGFuY2UocmF3LCAoc3RyLCBQYXRoKSk6CiAgICAgICAgcmF3ID0gUGF0aChyYXcpLnJlYWRfYnl0ZXMoKQogICAgcmV0dXJuIGhhc2hsaWIuc2hhMjU2KHJhdykuaGV4ZGlnZXN0KCkKCgpkZWYgc2FmZV9uYW1lKG5hbWUpOgogICAgaWYgbm90IG5hbWUgb3IgIlxcIiBpbiBuYW1lIG9yICI6IiBpbiBuYW1lIG9yICJceDAwIiBpbiBuYW1lOgogICAgICAgIHJhaXNlIFZhbHVlRXJyb3IoIlVuc2FmZSB0ZXN0IHBhY2thZ2UgcGF0aCIpCiAgICBwID0gUHVyZVBvc2l4UGF0aChuYW1lKQogICAgaWYgcC5pc19hYnNvbHV0ZSgpIG9yIGFueSh4IGluICgiIiwgIi4iLCAiLi4iKSBmb3IgeCBpbiBuYW1lLnNwbGl0KCIvIikpOgogICAgICAgIHJhaXNlIFZhbHVlRXJyb3IoIlVuc2FmZSB0ZXN0IHBhY2thZ2UgcGF0aCIpCiAgICByZXR1cm4gbmFtZQoKCmRlZiByZWFkX3ppcChwYXRoKToKICAgIHdpdGggemlwZmlsZS5aaXBGaWxlKHBhdGgpIGFzIGFyY2hpdmU6CiAgICAgICAgaWYgYXJjaGl2ZS50ZXN0emlwKCkgaXMgbm90IE5vbmU6IHJhaXNlIFZhbHVlRXJyb3IoIlRlc3QgcGFja2FnZSBDUkMgZmFpbHVyZSIpCiAgICAgICAgaW5mb3MgPSBhcmNoaXZlLmluZm9saXN0KCkKICAgICAgICBuYW1lcyA9IFtzYWZlX25hbWUoaS5maWxlbmFtZSkgZm9yIGkgaW4gaW5mb3MgaWYgbm90IGkuaXNfZGlyKCldCiAgICAgICAgaWYgbGVuKG5hbWVzKSAhPSBsZW4oc2V0KG5hbWVzKSk6IHJhaXNlIFZhbHVlRXJyb3IoIkR1cGxpY2F0ZSB0ZXN0IHBhY2thZ2UgbWVtYmVyIikKICAgICAgICBpZiBhbnkoKGkuZXh0ZXJuYWxfYXR0ciA+PiAxNikgJiAwbzE3MDAwMCA9PSAwbzEyMDAwMCBmb3IgaSBpbiBpbmZvcyk6CiAgICAgICAgICAgIHJhaXNlIFZhbHVlRXJyb3IoIlN5bWxpbmsgdGVzdCBwYWNrYWdlIG1lbWJlciIpCiAgICAgICAgcmV0dXJuIHtuYW1lOiBhcmNoaXZlLnJlYWQobmFtZSkgZm9yIG5hbWUgaW4gbmFtZXN9CgoKZGVmIGxvYWRfbWFuaWZlc3QocGF0aCwgZXhwZWN0ZWRfc2hhMjU2KToKICAgIHJhdyA9IFBhdGgocGF0aCkucmVhZF9ieXRlcygpCiAgICBpZiBzaGEocmF3KSAhPSBleHBlY3RlZF9zaGEyNTY6IHJhaXNlIFZhbHVlRXJyb3IoIlRlc3QgcHJvbXB0IG1hbmlmZXN0IGhhc2ggbWlzbWF0Y2giKQogICAgcm9vdCA9IGpzb24ubG9hZHMocmF3KQogICAgaWYgcm9vdC5nZXQoInN0YXR1cyIpICE9ICJTRUFMRURfVEVTVF9ERUxJVkVSWSI6IHJhaXNlIFZhbHVlRXJyb3IoIlRlc3QgcGFja2FnZSBub3Qgc2VhbGVkIikKICAgIGlmIChyb290LmdldCgibWV0aG9kX2lkIiksIHJvb3QuZ2V0KCJhcm0iKSwgcm9vdC5nZXQoInNlZWQiKSwgcm9vdC5nZXQoImluZmVyZW5jZV9zcGxpdCIpKSAhPSAoCiAgICAgICAgICAgICJiMy12ZXJpZmllZC1wYXRjaC12MSIsICJCMyIsIDQyLCAidGVzdCIpOgogICAgICAgIHJhaXNlIFZhbHVlRXJyb3IoIlRlc3QgcGFja2FnZSBzY29wZSBtaXNtYXRjaCIpCiAgICBpZiByb290LmdldCgibGFiZWxzX2luX2thZ2dsZV9pbnB1dCIpIGlzIG5vdCBGYWxzZTogcmFpc2UgVmFsdWVFcnJvcigiVGFyZ2V0IGxhYmVscyBtdXN0IGJlIGhlbGQgb3V0IG9mIEthZ2dsZSBpbnB1dCIpCiAgICByZXR1cm4gcm9vdAoKCmRlZiB2YWxpZGF0ZV9hcmNoaXZlKHBhdGgsIHJvb3QpOgogICAgcmF3ID0gUGF0aChwYXRoKS5yZWFkX2J5dGVzKCkKICAgIGlmIHNoYShyYXcpICE9IHJvb3RbImthZ2dsZV9pbnB1dF9hcmNoaXZlX3NoYTI1NiJdOiByYWlzZSBWYWx1ZUVycm9yKCJUZXN0IGlucHV0IGFyY2hpdmUgY2hlY2tzdW0gbWlzbWF0Y2giKQogICAgbWVtYmVycyA9IHJlYWRfemlwKHBhdGgpCiAgICBpZiBzZXQobWVtYmVycykgIT0gc2V0KHJvb3RbImthZ2dsZV9pbnB1dF9tZW1iZXJzX3NoYTI1NiJdKTogcmFpc2UgVmFsdWVFcnJvcigiVGVzdCBpbnB1dCBwYWNrYWdlIGludmVudG9yeSBtaXNtYXRjaCIpCiAgICBmb3IgbmFtZSwgZXhwZWN0ZWQgaW4gcm9vdFsia2FnZ2xlX2lucHV0X21lbWJlcnNfc2hhMjU2Il0uaXRlbXMoKToKICAgICAgICBpZiBzaGEobWVtYmVyc1tuYW1lXSkgIT0gZXhwZWN0ZWQ6IHJhaXNlIFZhbHVlRXJyb3IoIlRlc3QgaW5wdXQgbWVtYmVyIGhhc2ggbWlzbWF0Y2g6ICIgKyBuYW1lKQogICAgaWYgYW55KHJlLnNlYXJjaChyIihefC8pKD86bGFiZWxzKD86XC5qc29ufFwuanNvbmwpP3x0ZXN0X2xhYmVsc1wuanNvbikkIiwgbiwgcmUuSSkgZm9yIG4gaW4gbWVtYmVycyk6CiAgICAgICAgcmFpc2UgVmFsdWVFcnJvcigiTGFiZWwgYXJ0aWZhY3QgZm91bmQgaW4gS2FnZ2xlIGluZmVyZW5jZSBwYWNrYWdlIikKICAgIHJldHVybiBtZW1iZXJzCgoKZGVmIHJlYWRfaW5wdXRfZGlyZWN0b3J5KHBhdGgpOgogICAgYmFzZSA9IFBhdGgocGF0aCkKICAgIGlmIG5vdCBiYXNlLmlzX2RpcigpOiByYWlzZSBWYWx1ZUVycm9yKCJFeHBhbmRlZCB0ZXN0IGlucHV0IGRpcmVjdG9yeSBpcyBtaXNzaW5nIikKICAgIG1lbWJlcnMgPSB7fQogICAgZm9yIGl0ZW0gaW4gc29ydGVkKGJhc2Uucmdsb2IoIioiKSk6CiAgICAgICAgaWYgaXRlbS5pc19zeW1saW5rKCk6IHJhaXNlIFZhbHVlRXJyb3IoIlN5bWxpbmsgaW4gZXhwYW5kZWQgdGVzdCBpbnB1dCIpCiAgICAgICAgaWYgaXRlbS5pc19maWxlKCk6CiAgICAgICAgICAgIG5hbWUgPSBpdGVtLnJlbGF0aXZlX3RvKGJhc2UpLmFzX3Bvc2l4KCkKICAgICAgICAgICAgc2FmZV9uYW1lKG5hbWUpCiAgICAgICAgICAgIGlmIG5hbWUgaW4gbWVtYmVyczogcmFpc2UgVmFsdWVFcnJvcigiRHVwbGljYXRlIGV4cGFuZGVkIHRlc3QgaW5wdXQgbWVtYmVyIikKICAgICAgICAgICAgbWVtYmVyc1tuYW1lXSA9IGl0ZW0ucmVhZF9ieXRlcygpCiAgICByZXR1cm4gbWVtYmVycwoKCmRlZiB2YWxpZGF0ZV9tZW1iZXJzKG1lbWJlcnMsIHJvb3QpOgogICAgaWYgc2V0KG1lbWJlcnMpICE9IHNldChyb290WyJrYWdnbGVfaW5wdXRfbWVtYmVyc19zaGEyNTYiXSk6CiAgICAgICAgcmFpc2UgVmFsdWVFcnJvcigiVGVzdCBpbnB1dCBtZW1iZXIgaW52ZW50b3J5IG1pc21hdGNoIikKICAgIGZvciBuYW1lLCBleHBlY3RlZCBpbiByb290WyJrYWdnbGVfaW5wdXRfbWVtYmVyc19zaGEyNTYiXS5pdGVtcygpOgogICAgICAgIGlmIHNoYShtZW1iZXJzW25hbWVdKSAhPSBleHBlY3RlZDoKICAgICAgICAgICAgcmFpc2UgVmFsdWVFcnJvcigiVGVzdCBpbnB1dCBtZW1iZXIgY2hlY2tzdW0gbWlzbWF0Y2g6ICIgKyBuYW1lKQogICAgaWYgYW55KHJlLnNlYXJjaChyIihefC8pKD86bGFiZWxzKD86XC5qc29ufFwuanNvbmwpP3x0ZXN0X2xhYmVsc1wuanNvbikkIiwgbiwgcmUuSSkgZm9yIG4gaW4gbWVtYmVycyk6CiAgICAgICAgcmFpc2UgVmFsdWVFcnJvcigiTGFiZWwgYXJ0aWZhY3QgZm91bmQgaW4gS2FnZ2xlIGluZmVyZW5jZSBwYWNrYWdlIikKICAgIHJldHVybiBtZW1iZXJzCgoKZGVmIHJlYWRfZGF0YXNldF9yb3dzKG1lbWJlcnMsIHJvb3QsIGRhdGFzZXQpOgogICAgaW5mbyA9IHJvb3RbImRhdGFzZXRzIl1bZGF0YXNldF0KICAgIG5hbWUgPSBmInByb21wdHMve2RhdGFzZXR9Lmpzb25sIgogICAgcmF3ID0gbWVtYmVyc1tuYW1lXQogICAgaWYgc2hhKHJhdykgIT0gaW5mb1sicHJvbXB0c19zaGEyNTYiXTogcmFpc2UgVmFsdWVFcnJvcigiUHJvbXB0IHNldCBoYXNoIG1pc21hdGNoIikKICAgIHJvd3MgPSBbanNvbi5sb2FkcyhsaW5lKSBmb3IgbGluZSBpbiByYXcuZGVjb2RlKCJ1dGYtOCIpLnNwbGl0bGluZXMoKSBpZiBsaW5lLnN0cmlwKCldCiAgICBpZiBsZW4ocm93cykgIT0gaW5mb1siZXhwZWN0ZWQiXSBvciBsZW4oe3IuZ2V0KCJpZCIpIGZvciByIGluIHJvd3N9KSAhPSBsZW4ocm93cyk6CiAgICAgICAgcmFpc2UgVmFsdWVFcnJvcigiVGVzdCBjZW5zdXMgaGFzIG1pc3NpbmcvZHVwbGljYXRlIElEcyIpCiAgICBmb3Igcm93IGluIHJvd3M6CiAgICAgICAgaWYgYW55KGsgaW4gcm93IGZvciBrIGluICgibGFiZWwiLCAidGFyZ2V0IiwgInRhcmdldF9sYWJlbCIpKToKICAgICAgICAgICAgcmFpc2UgVmFsdWVFcnJvcigiVGFyZ2V0IGxhYmVsIGxlYWtlZCBpbnRvIHRlc3QgcHJvbXB0IHJvdyIpCiAgICAgICAgaWYgKHJvdy5nZXQoImRhdGFzZXQiKSwgcm93LmdldCgic3BsaXQiKSwgcm93LmdldCgibWV0aG9kX2lkIikpICE9IChkYXRhc2V0LCAidGVzdCIsIHJvb3RbIm1ldGhvZF9pZCJdKToKICAgICAgICAgICAgcmFpc2UgVmFsdWVFcnJvcigiVGVzdCBwcm9tcHQgc2NvcGUgbWlzbWF0Y2giKQogICAgICAgIGlmIG5vdCByb3dbImlkIl0uc3RhcnRzd2l0aChkYXRhc2V0ICsgIjp0ZXN0OiIpIG9yIHNoYShyb3dbInByb21wdCJdLmVuY29kZSgidXRmLTgiKSkgIT0gcm93WyJwcm9tcHRfc2hhMjU2Il06CiAgICAgICAgICAgIHJhaXNlIFZhbHVlRXJyb3IoIlRlc3QgcHJvbXB0IElEL2hhc2ggbWlzbWF0Y2giKQogICAgcmV0dXJuIHJvd3MKCgpkZWYgbG9hZF90aHJlc2hvbGQocm9vdCwgbWVtYmVycywgZGF0YXNldCk6CiAgICAiIiJWZXJpZnkgdmFsaWRhdGlvbi1maXR0ZWQgdGhyZXNob2xkIGJ5dGVzIGFuZCB0aGVpciBmcm96ZW4gbWV0aG9kIGlkZW50aXR5LiIiIgogICAgbmFtZSA9IGYidGhyZXNob2xkX2xvY2tzL3tkYXRhc2V0fV90aHJlc2hvbGRfbG9jay5qc29uIgogICAgcmF3ID0gbWVtYmVyc1tuYW1lXQogICAgaWYgc2hhKHJhdykgIT0gcm9vdFsidGhyZXNob2xkX2xvY2tfc2hhMjU2Il1bZGF0YXNldF06CiAgICAgICAgcmFpc2UgVmFsdWVFcnJvcigiVGhyZXNob2xkIGxvY2sgZmlsZSBjaGVja3N1bSBtaXNtYXRjaCIpCiAgICBsb2NrID0ganNvbi5sb2FkcyhyYXcuZGVjb2RlKCJ1dGYtOCIpKQogICAgdW5zaWduZWQgPSB7azogdiBmb3IgaywgdiBpbiBsb2NrLml0ZW1zKCkgaWYgayAhPSAic2VhbF9zaGEyNTYifQogICAgY2Fub25pY2FsID0ganNvbi5kdW1wcyh1bnNpZ25lZCwgc29ydF9rZXlzPVRydWUsIHNlcGFyYXRvcnM9KCIsIiwgIjoiKSwKICAgICAgICAgICAgICAgICAgICAgICAgICAgZW5zdXJlX2FzY2lpPUZhbHNlLCBhbGxvd19uYW49RmFsc2UpLmVuY29kZSgidXRmLTgiKQogICAgaWYgc2hhKGNhbm9uaWNhbCkgIT0gbG9jay5nZXQoInNlYWxfc2hhMjU2Iik6CiAgICAgICAgcmFpc2UgVmFsdWVFcnJvcigiVGhyZXNob2xkIGxvY2sgY2Fub25pY2FsIHNlYWwgbWlzbWF0Y2giKQogICAgaWYgKGxvY2suZ2V0KCJzdGF0dXMiKSwgbG9jay5nZXQoImFybSIpLCBsb2NrLmdldCgibWV0aG9kX2lkIiksCiAgICAgICAgbG9jay5nZXQoInNlZWQiKSwgbG9jay5nZXQoImRhdGFzZXQiKSwgbG9jay5nZXQoImZpdF9zcGxpdCIpKSAhPSAoCiAgICAgICAgIkxPQ0tFRCIsICJCMyIsIHJvb3RbIm1ldGhvZF9pZCJdLCA0MiwgZGF0YXNldCwgInZhbGlkYXRpb24iKToKICAgICAgICByYWlzZSBWYWx1ZUVycm9yKCJUaHJlc2hvbGQgc2NvcGUgbWlzbWF0Y2giKQogICAgZXhwZWN0ZWQgPSB7Im1ldGhvZF9pZCI6IHJvb3RbIm1ldGhvZF9pZCJdLAogICAgICAgICJwb29sX3NoYTI1NiI6IHJvb3RbImRhdGFzZXRzIl1bZGF0YXNldF1bInBvb2xfc2hhMjU2Il0sCiAgICAgICAgIm1vZGVsX3NoYTI1NiI6IHJvb3RbIm1vZGVsX3NoYTI1NiJdLAogICAgICAgICJ0b2tlbml6ZXJfcmV2aXNpb24iOiByb290WyJ0b2tlbml6ZXJfcmV2aXNpb24iXSwKICAgICAgICAicnVudGltZV9jb250cmFjdF9zaGEyNTYiOiByb290WyJ0aHJlc2hvbGRfdmFsaWRhdGlvbl9ydW50aW1lX2NvbnRyYWN0X3NoYTI1NiJdLAogICAgICAgICJzY29yaW5nIjogInVuY29uc3RyYWluZWRfZmlyc3RfdG9rZW5fbG9naXRzX3YxIiwKICAgICAgICAid29ya2VyX3NoYTI1NiI6IHJvb3RbInRocmVzaG9sZF9hcHBsaWNhdGlvbl93b3JrZXJfc2hhMjU2Il19CiAgICBpZiBhbnkobG9ja1siYXBwbGljYXRpb25faWRlbnRpdHkiXS5nZXQoaykgIT0gdiBmb3IgaywgdiBpbiBleHBlY3RlZC5pdGVtcygpKToKICAgICAgICByYWlzZSBWYWx1ZUVycm9yKCJUaHJlc2hvbGQgYXBwbGljYXRpb24gaWRlbnRpdHkgbWlzbWF0Y2giKQogICAgdGF1ID0gbG9jay5nZXQoInRhdV9zdGFyIikKICAgIGlmIHR5cGUodGF1KSBub3QgaW4gKGZsb2F0LCBpbnQpIG9yIG5vdCAwLjAxIDw9IHRhdSA8PSAwLjk5OgogICAgICAgIHJhaXNlIFZhbHVlRXJyb3IoIkludmFsaWQgbG9ja2VkIHRocmVzaG9sZCIpCiAgICBpZiB0YXUgIT0gcm9vdFsiZGF0YXNldHMiXVtkYXRhc2V0XVsidGF1X3N0YXIiXToKICAgICAgICByYWlzZSBWYWx1ZUVycm9yKCJUaHJlc2hvbGQgdmFsdWUgZGlmZmVycyBmcm9tIHNlYWxlZCB0ZXN0IGRlbGl2ZXJ5IikKICAgIHJldHVybiBmbG9hdCh0YXUpCgoKZGVmIHJ1bl9pZGVudGl0eShyb290LCBkYXRhc2V0KToKICAgIHggPSByb290WyJkYXRhc2V0cyJdW2RhdGFzZXRdCiAgICByZXR1cm4geyJyZWxlYXNlX2lkIjogcm9vdFsicmVsZWFzZV9pZCJdLCAibWV0aG9kX2lkIjogcm9vdFsibWV0aG9kX2lkIl0sICJhcm0iOiAiQjMiLAogICAgICAgICAgICAiZGF0YXNldCI6IGRhdGFzZXQsICJzcGxpdCI6ICJ0ZXN0IiwgInNlZWQiOiA0MiwgImNvdW50IjogeFsiZXhwZWN0ZWQiXSwKICAgICAgICAgICAgInByb21wdHNfc2hhMjU2IjogeFsicHJvbXB0c19zaGEyNTYiXSwgInBvb2xfc2hhMjU2IjogeFsicG9vbF9zaGEyNTYiXSwKICAgICAgICAgICAgIm1vZGVsX3NoYTI1NiI6IHJvb3RbIm1vZGVsX3NoYTI1NiJdLCAidG9rZW5pemVyX3JldmlzaW9uIjogcm9vdFsidG9rZW5pemVyX3JldmlzaW9uIl0sCiAgICAgICAgICAgICJydW50aW1lX2NvbnRyYWN0X3NoYTI1NiI6IHJvb3RbInRocmVzaG9sZF92YWxpZGF0aW9uX3J1bnRpbWVfY29udHJhY3Rfc2hhMjU2Il0sCiAgICAgICAgICAgICJ0aHJlc2hvbGRfbG9ja19zaGEyNTYiOiByb290WyJ0aHJlc2hvbGRfbG9ja19zaGEyNTYiXVtkYXRhc2V0XSwKICAgICAgICAgICAgIndvcmtlcl9zaGEyNTYiOiByb290WyJ0ZXN0X3dvcmtlcl9zaGEyNTYiXX0KCgpkZWYgY2hlY2tlZF9yZWNvcmRzKG91dHB1dF9yb290LCByb290LCByb3dzLCBkYXRhc2V0LCBzcGxpdD0idGVzdCIsIHNlZWQ9NDIsIGFybT0iQjMiKToKICAgIGJhc2UgPSBQYXRoKG91dHB1dF9yb290KSAvIGYic2VlZF97c2VlZH0iIC8gZGF0YXNldCAvIHNwbGl0CiAgICBwYXRoID0gYmFzZSAvICJyZWNvcmRzLmpzb25sIgogICAgaWYgbm90IHBhdGguaXNfZmlsZSgpOiByZXR1cm4gMAogICAgbWFuaWZlc3RfcGF0aCA9IGJhc2UgLyAicnVuX21hbmlmZXN0Lmpzb24iCiAgICBpZiBub3QgbWFuaWZlc3RfcGF0aC5pc19maWxlKCk6IHJhaXNlIFZhbHVlRXJyb3IoIlRlc3QgcmVjb3JkcyBsYWNrIHJ1biBpZGVudGl0eSBtYW5pZmVzdCIpCiAgICBtYW5pZmVzdCA9IGpzb24ubG9hZHMobWFuaWZlc3RfcGF0aC5yZWFkX3RleHQoZW5jb2Rpbmc9InV0Zi04IikpCiAgICBleHBlY3RlZF9pZGVudGl0eSA9IHJ1bl9pZGVudGl0eShyb290LCBkYXRhc2V0KQogICAgaWYgbWFuaWZlc3QuZ2V0KCJiM190ZXN0X2lkZW50aXR5IikgIT0gZXhwZWN0ZWRfaWRlbnRpdHk6IHJhaXNlIFZhbHVlRXJyb3IoIlRlc3QgcmVzdW1lIGlkZW50aXR5IG1pc21hdGNoIikKICAgIGxvb2t1cCA9IHtyWyJpZCJdOiByIGZvciByIGluIHJvd3N9CiAgICBzZWVuID0gc2V0KCkKICAgIGZvciBsaW5lIGluIHBhdGgucmVhZF90ZXh0KGVuY29kaW5nPSJ1dGYtOCIpLnNwbGl0bGluZXMoKToKICAgICAgICBpZiBub3QgbGluZS5zdHJpcCgpOiBjb250aW51ZQogICAgICAgIHJlY29yZCA9IGpzb24ubG9hZHMobGluZSkKICAgICAgICB1aWQgPSByZWNvcmRbImlkIl0KICAgICAgICBpZiB1aWQgaW4gc2VlbiBvciB1aWQgbm90IGluIGxvb2t1cDogcmFpc2UgVmFsdWVFcnJvcigiRHVwbGljYXRlL3VuZXhwZWN0ZWQgdGVzdCBJRCIpCiAgICAgICAgaWYgYW55KGsgaW4gcmVjb3JkIGZvciBrIGluICgibGFiZWwiLCAidGFyZ2V0IiwgInRhcmdldF9sYWJlbCIpKToKICAgICAgICAgICAgcmFpc2UgVmFsdWVFcnJvcigiVGFyZ2V0IGxhYmVsIGxlYWtlZCBpbnRvIHRlc3QgcHJlZGljdGlvbiByZWNvcmQiKQogICAgICAgIHByb21wdCA9IGxvb2t1cFt1aWRdCiAgICAgICAgaWYgKHJlY29yZC5nZXQoInByb21wdF9zaGEyNTYiKSwgcmVjb3JkLmdldCgiZGF0YXNldCIpLCByZWNvcmQuZ2V0KCJzcGxpdCIpLAogICAgICAgICAgICAgICAgcmVjb3JkLmdldCgiYXJtIiksIHJlY29yZC5nZXQoInNlZWQiKSwgcmVjb3JkLmdldCgidmFsaWQiKSwKICAgICAgICAgICAgICAgIHJlY29yZC5nZXQoIm1vZGVsX3NoYTI1NiIpLCByZWNvcmQuZ2V0KCJydW50aW1lX2NvbnRyYWN0X3NoYTI1NiIpKSAhPSAoCiAgICAgICAgICAgICAgICBwcm9tcHRbInByb21wdF9zaGEyNTYiXSwgZGF0YXNldCwgInRlc3QiLCBhcm0sIHNlZWQsIFRydWUsCiAgICAgICAgICAgICAgICByb290WyJtb2RlbF9zaGEyNTYiXSwgcm9vdFsicnVudGltZV9jb250cmFjdF9zaGEyNTYiXSk6CiAgICAgICAgICAgIHJhaXNlIFZhbHVlRXJyb3IoIlRlc3QgcHJlZGljdGlvbiBpZGVudGl0eSBtaXNtYXRjaCIpCiAgICAgICAgcywgbHAwLCBscDEgPSByZWNvcmRbInMiXSwgcmVjb3JkWyJsb2dwcm9iXzAiXSwgcmVjb3JkWyJsb2dwcm9iXzEiXQogICAgICAgIGlmIGFueSh0eXBlKHgpIG5vdCBpbiAoaW50LCBmbG9hdCkgb3Igbm90IG1hdGguaXNmaW5pdGUoeCkgZm9yIHggaW4gKHMsIGxwMCwgbHAxKSkgb3Igbm90IDAgPD0gcyA8PSAxOgogICAgICAgICAgICByYWlzZSBWYWx1ZUVycm9yKCJJbnZhbGlkIHRlc3Qgc2NvcmUiKQogICAgICAgICMgVXNlIGEgc3RhYmxlIGJyYW5jaC1mb3JtIHNpZ21vaWQgd2hlbiB0aGUgaG9zdCBtYXRoIG1vZHVsZSBsYWNrcyBsb2dhZGRleHAuCiAgICAgICAgZGVsdGEgPSBscDEgLSBscDAKICAgICAgICBleHBlY3RlZCA9IDEgLyAoMSArIG1hdGguZXhwKC1kZWx0YSkpIGlmIGRlbHRhID49IDAgZWxzZSBtYXRoLmV4cChkZWx0YSkgLyAoMSArIG1hdGguZXhwKGRlbHRhKSkKICAgICAgICBpZiBhYnMocyAtIGV4cGVjdGVkKSA+IDFlLTEyOiByYWlzZSBWYWx1ZUVycm9yKCJUZXN0IHNjb3JlL2xvZ3Byb2IgbWlzbWF0Y2giKQogICAgICAgIHRhdSA9IHJvb3RbImRhdGFzZXRzIl1bZGF0YXNldF1bInRhdV9zdGFyIl0KICAgICAgICBpZiByZWNvcmQuZ2V0KCJ0aHJlc2hvbGRfdGF1IikgIT0gdGF1IG9yIHJlY29yZC5nZXQoInRocmVzaG9sZF9wcmVkaWN0aW9uIikgIT0gaW50KHMgPj0gdGF1KToKICAgICAgICAgICAgcmFpc2UgVmFsdWVFcnJvcigiVGVzdCBsb2NrZWQtdGhyZXNob2xkIHByZWRpY3Rpb24gbWlzbWF0Y2giKQogICAgICAgIHNlZW4uYWRkKHVpZCkKICAgIHJldHVybiBsZW4oc2VlbikK', 'kaggle_inference_test.py': '...'}.items(): (WORK/name).write_bytes(base64.b64decode(encoded))
sys.path.insert(0,str(WORK))


In [2]:
import glob, os, json, hashlib
from pathlib import Path
from grace_runtime_contract import load_contract, assert_runtime, digest, command
RUNTIME=load_contract()
EXPECTED_MANIFEST_SHA256='62a78ff8853373b0fae261b95ae52b4ff117223a5061d77f9c44f1ae64551127'
INPUT_ROOT=Path('/kaggle/input')
DATASETS_ROOT=INPUT_ROOT/'datasets'
EXPECTED_DATASET_ROOT=DATASETS_ROOT/'lznt123'/'grace-b3-test-runtime-lock-20261110-v1'
input_roots=sorted(Path(x) for x in glob.glob('/kaggle/input/*') if Path(x).is_dir())
# Kaggle's nested input mounts can be symlinked; pathlib.rglob() does not
# reliably descend through those mounts. Probe the exact pinned mount first,
# then walk only the datasets mount with followlinks=True.
manifest_candidates=[]
expected_manifest=EXPECTED_DATASET_ROOT/'test_release_manifest.json'
if expected_manifest.is_file():
 manifest_candidates=[expected_manifest]
else:
 visited_dirs=set()
 for current, dirs, files in os.walk(DATASETS_ROOT, topdown=True, followlinks=True):
  real=os.path.realpath(current)
  if real in visited_dirs:
   dirs[:]=[]
   continue
  visited_dirs.add(real)
  dirs.sort()
  if 'test_release_manifest.json' in files:
   manifest_candidates.append(Path(current)/'test_release_manifest.json')
 manifest_candidates=sorted({p.resolve() for p in manifest_candidates})
if len(manifest_candidates)!=1:
 mount_details={}
 for root in input_roots:
  try: mount_details[str(root)]={'is_symlink':root.is_symlink(),'resolved':str(root.resolve()),'children':sorted(p.name for p in root.iterdir())[:30]}
  except Exception as exc: mount_details[str(root)]={'error':repr(exc)}
 print(json.dumps({'preflight':'FAIL','input_roots':[str(x) for x in input_roots],'mount_details':mount_details,'expected_manifest':str(expected_manifest),'expected_manifest_exists':expected_manifest.is_file(),'manifest_candidates':[str(x) for x in manifest_candidates]},indent=2))
 raise RuntimeError('Expected exactly one B3 test manifest; see mount diagnostics above')
MANIFEST_PATH=manifest_candidates[0]
DATASET_ROOT=MANIFEST_PATH.parent
if hashlib.sha256(MANIFEST_PATH.read_bytes()).hexdigest()!=EXPECTED_MANIFEST_SHA256:
 raise RuntimeError('Test release manifest checksum mismatch')
from b3_test_runtime import load_manifest, read_input_directory, validate_members, validate_archive, read_dataset_rows, load_threshold
ROOT=load_manifest(MANIFEST_PATH,EXPECTED_MANIFEST_SHA256)
expanded_input=DATASET_ROOT/'kaggle_input'
archive_input=DATASET_ROOT/'kaggle_input.zip'
if expanded_input.is_dir():
 INPUT_DIR=expanded_input
 MEMBERS=validate_members(read_input_directory(INPUT_DIR),ROOT)
 INPUT_SOURCE=INPUT_DIR
elif archive_input.is_file():
 MEMBERS=validate_archive(archive_input,ROOT)
 INPUT_DIR=WORK/'b3_test_input_expanded'
 if INPUT_DIR.exists(): raise RuntimeError('Existing extracted test input requires verified resume; refusing overwrite')
 INPUT_DIR.mkdir(parents=True)
 for member_name, member_bytes in MEMBERS.items():
  parts=member_name.split('/')
  if not parts or any(part in ('','.','..') for part in parts) or '\\' in member_name or ':' in member_name:
   raise RuntimeError('Unsafe path in validated test archive: '+member_name)
  target=INPUT_DIR.joinpath(*parts)
  target.parent.mkdir(parents=True,exist_ok=True)
  target.write_bytes(member_bytes)
 MEMBERS=validate_members(read_input_directory(INPUT_DIR),ROOT)
 INPUT_SOURCE=archive_input
else:
 print(json.dumps({'preflight':'FAIL','manifest':str(MANIFEST_PATH),'dataset_root':str(DATASET_ROOT),'expanded_input_exists':expanded_input.is_dir(),'archive_exists':archive_input.is_file(),'input_root_entries':[x.name for x in DATASET_ROOT.iterdir()]},indent=2))
 raise RuntimeError('Neither expanded kaggle_input/ nor pinned kaggle_input.zip exists beside the manifest')
model_filename=RUNTIME['model']['filename']
model_candidates=sorted({p.resolve() for p in INPUT_ROOT.rglob(model_filename) if p.is_file()})
preferred_model=Path('/kaggle/input/notebooks/lznt123/qwen3coder30b-prepare-notebook/qwen_gguf')/model_filename
if preferred_model.is_file():
 MODEL_PATH=preferred_model.resolve()
elif len(model_candidates)==1:
 MODEL_PATH=model_candidates[0]
else:
 print(json.dumps({'preflight':'FAIL','manifest':str(MANIFEST_PATH),'input_source':str(INPUT_SOURCE),'model_filename':model_filename,'preferred_model_exists':preferred_model.is_file(),'model_candidates':[str(x) for x in model_candidates],'input_roots':[str(x) for x in input_roots]},indent=2))
 raise RuntimeError('Could not resolve exactly one pinned model; see candidate paths above')
print(json.dumps({'preflight':'PASS','manifest':str(MANIFEST_PATH),'input_source':str(INPUT_SOURCE),'model_path':str(MODEL_PATH),'model_candidate_count':len(model_candidates)},indent=2))
if hashlib.sha256(MODEL_PATH.read_bytes()).hexdigest()!=RUNTIME['model']['sha256']: raise RuntimeError('Model checkpoint checksum mismatch')
if hashlib.sha256(Path('/kaggle/working/kaggle_inference_test.py').read_bytes()).hexdigest()!=ROOT['test_worker_sha256']: raise RuntimeError('Test worker source hash mismatch')
if hashlib.sha256(MEMBERS[ROOT['runtime_lock_member']]).hexdigest()!=ROOT['runtime_lock_sha256']: raise RuntimeError('Runtime contract hash mismatch')
if ROOT['runtime_lock_sha256']!=hashlib.sha256(Path('/kaggle/working/inference_runtime_lock.json').read_bytes()).hexdigest(): raise RuntimeError('Notebook/runtime lock differs from release lock')
for ds in ('devign','reveal'):
 rows=read_dataset_rows(MEMBERS,ROOT,ds)
 tau=load_threshold(ROOT,MEMBERS,ds)
 print(f'Preflight {ds}: n={len(rows)} threshold={tau}')
gpu=command(['nvidia-smi','--query-gpu=index,uuid,name,memory.free,memory.total,driver_version','--format=csv,noheader'])
if len([x for x in gpu.splitlines() if x.strip()])!=RUNTIME['gpu']['device_count'] or any('T4' not in x for x in gpu.splitlines()): raise RuntimeError('Runtime contract requires 2x Tesla T4')


{
  "preflight": "PASS",
  "manifest": "/kaggle/input/datasets/lznt123/grace-b3-test-runtime-lock-20261110-v1/test_release_manifest.json",
  "input_source": "/kaggle/input/datasets/lznt123/grace-b3-test-runtime-lock-20261110-v1/kaggle_input",
  "model_path": "/kaggle/input/notebooks/duchuynguyen1802/huy-qwen3coder30b-prepare-notebook/qwen_gguf/Qwen3-Coder-30B-A3B-Instruct-Q4_K_M.gguf",
  "model_candidate_count": 1
}
Preflight devign: n=2732 threshold=0.99
Preflight reveal: n=2274 threshold=0.01


In [3]:
import importlib.metadata, sys, urllib.request, shutil, os
build=RUNTIME['build']
os.environ['CMAKE_ARGS']=(f"-DGGML_CUDA={'on' if build['GGML_CUDA'] else 'off'}" f" -DCMAKE_CUDA_ARCHITECTURES={build['CMAKE_CUDA_ARCHITECTURES']}" f" -DCMAKE_LIBRARY_PATH={build['CMAKE_LIBRARY_PATH']}")
for key in ('CMAKE_BUILD_PARALLEL_LEVEL','FORCE_CMAKE'): os.environ[key]=build[key]
packages=[f'{n}=={v}' for n,v in RUNTIME['packages'].items() if n!=RUNTIME['backend_source']['name']]
packages += ['typing-extensions>=4.5.0','diskcache>=5.6.1','jinja2>=2.11.3']
diag=Path('/kaggle/working/grace_diagnostics'); diag.mkdir(exist_ok=True)
with (diag/'install_dependencies.log').open('w',encoding='utf-8') as log:
 subprocess.run([sys.executable,'-m','pip','install','--no-cache-dir',*packages],check=True,stdout=log,stderr=subprocess.STDOUT)
source=RUNTIME['backend_source']
with urllib.request.urlopen(f"https://pypi.org/pypi/{source['name']}/{source['version']}/json",timeout=60) as response: meta=json.load(response)
artifact=next(x for x in meta['urls'] if x['filename']==source['sdist_filename'])
if artifact['digests']['sha256']!=source['sdist_sha256']: raise RuntimeError('llama.cpp source registry hash mismatch')
archive=Path('/kaggle/working')/source['sdist_filename']
with urllib.request.urlopen(artifact['url'],timeout=120) as response, archive.open('wb') as target: shutil.copyfileobj(response,target)
if digest(archive)!=source['sdist_sha256']: raise RuntimeError('llama.cpp source archive hash mismatch')
with (diag/'install_llama_cuda.log').open('w',encoding='utf-8') as log:
 subprocess.run([sys.executable,'-m','pip','install','-v','--no-cache-dir','--no-deps','--force-reinstall',str(archive)],check=True,stdout=log,stderr=subprocess.STDOUT)
assert_runtime(RUNTIME)


({'huggingface-hub': '0.36.2',
  'llama-cpp-python': '0.3.16',
  'numpy': '2.1.3',
  'scikit-learn': '1.5.2',
  'scipy': '1.16.3',
  'tokenizers': '0.22.2',
  'transformers': '4.57.1'},
 '0, GPU-0fb00d20-6fa8-e5f1-1bf1-9155a58f3c82, Tesla T4, 14912 MiB, 15360 MiB, 580.178.04\n1, GPU-a95bd993-92fb-c92f-1aaf-2ca346999638, Tesla T4, 14912 MiB, 15360 MiB, 580.178.04\n')

In [4]:
assert_runtime(RUNTIME)
import llama_cpp
assert llama_cpp.llama_supports_gpu_offload(), 'Locked CUDA backend is unavailable'
print('Locked package versions:', {n:importlib.metadata.version(n) for n in RUNTIME['packages']})
print(command(['nvidia-smi','--query-gpu=index,uuid,name,memory.free,memory.total,driver_version','--format=csv,noheader']))


ggml_cuda_init: GGML_CUDA_FORCE_MMQ:    no
ggml_cuda_init: GGML_CUDA_FORCE_CUBLAS: no
ggml_cuda_init: found 2 CUDA devices:
  Device 0: Tesla T4, compute capability 7.5, VMM: yes
  Device 1: Tesla T4, compute capability 7.5, VMM: yes


Locked package versions: {'huggingface-hub': '0.36.2', 'llama-cpp-python': '0.3.16', 'numpy': '2.1.3', 'scikit-learn': '1.5.2', 'scipy': '1.16.3', 'tokenizers': '0.22.2', 'transformers': '4.57.1'}
0, GPU-0fb00d20-6fa8-e5f1-1bf1-9155a58f3c82, Tesla T4, 14909 MiB, 15360 MiB, 580.178.04
1, GPU-a95bd993-92fb-c92f-1aaf-2ca346999638, Tesla T4, 14807 MiB, 15360 MiB, 580.178.04



In [5]:
import os, sys, time, subprocess
OUT=Path('/kaggle/working/b3_test_release_20261110')
if OUT.exists(): raise RuntimeError('Existing output requires verified resume; refusing overwrite')
env=os.environ.copy()
env.update(B3_TEST_MANIFEST_PATH=str(MANIFEST_PATH),B3_TEST_MANIFEST_SHA256=EXPECTED_MANIFEST_SHA256,B3_TEST_INPUT_DIR=str(INPUT_DIR))
status=OUT/'status.json'
cmd=[sys.executable,'/kaggle/working/kaggle_inference_test.py','--model',str(MODEL_PATH),'--seed','42','--arm','B3',
 '--expected-model-sha256',RUNTIME['model']['sha256'],'--model-revision','local-input','--gpu-layers','-1',
 '--deadline-epoch',str(SESSION_STARTED+10*3600),'--checkpoint-reserve-seconds','300','--status',str(status),
 '--run','devign','test','devign',str(OUT/'seed_42/devign/test'),
 '--run','reveal','test','reveal',str(OUT/'seed_42/reveal/test')]
subprocess.run(cmd,check=True,env=env)


ggml_cuda_init: GGML_CUDA_FORCE_MMQ:    no
ggml_cuda_init: GGML_CUDA_FORCE_CUBLAS: no
ggml_cuda_init: found 2 CUDA devices:
  Device 0: Tesla T4, compute capability 7.5, VMM: yes
  Device 1: Tesla T4, compute capability 7.5, VMM: yes


llama_model_load_from_file_impl: using device CUDA0 (Tesla T4) - 14806 MiB free
llama_model_load_from_file_impl: using device CUDA1 (Tesla T4) - 14704 MiB free
llama_model_loader: loaded meta data with 39 key-value pairs and 579 tensors from /kaggle/input/notebooks/duchuynguyen1802/huy-qwen3coder30b-prepare-notebook/qwen_gguf/Qwen3-Coder-30B-A3B-Instruct-Q4_K_M.gguf (version GGUF V3 (latest))
llama_model_loader: Dumping metadata keys/values. Note: KV overrides do not apply in this output.
llama_model_loader: - kv   0:                       general.architecture str              = qwen3moe
llama_model_loader: - kv   1:                               general.type str              = model
llama_model_loader: - kv   2:                     general.sampling.top_k i32              = 20
llama_model_loader: - kv   3:                     general.sampling.top_p f32              = 0.800000
llama_model_loader: - kv   4:                      general.sampling.temp f32              = 0.700000
llama_mode

CompletedProcess(args=['/usr/bin/python3', '/kaggle/working/kaggle_inference_test.py', '--model', '/kaggle/input/notebooks/duchuynguyen1802/huy-qwen3coder30b-prepare-notebook/qwen_gguf/Qwen3-Coder-30B-A3B-Instruct-Q4_K_M.gguf', '--seed', '42', '--arm', 'B3', '--expected-model-sha256', 'f6050b4da22869aa3fb9d94bb3314e9fc5a027d4741c1f97761c387d6b2b2335', '--model-revision', 'local-input', '--gpu-layers', '-1', '--deadline-epoch', '1794338999.3411279', '--checkpoint-reserve-seconds', '300', '--status', '/kaggle/working/b3_test_release_20261110/status.json', '--run', 'devign', 'test', 'devign', '/kaggle/working/b3_test_release_20261110/seed_42/devign/test', '--run', 'reveal', 'test', 'reveal', '/kaggle/working/b3_test_release_20261110/seed_42/reveal/test'], returncode=0)